# Phase 3.3 : Fine-tuning des dernières couches

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase3_2_head_training.ipynb`.

**Ce notebook est autonome** : la section "Reprise" ci-dessous refait le setup des phases 1.1, 1.2, 3.1 et 3.2 (téléchargement, organisation, pipeline MobileNetV2, base gelée + tête custom, **entraînement de la tête**), sans les explications déjà vues. Le fine-tuning ne fait sens qu'après cet entraînement préalable de la tête.

## Reprise (setup phases 1.1, 1.2, 3.1 et 3.2, condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ne suffisent : ce dataset
    # contient quelques fichiers que seul le decodeur TensorFlow rejette au moment de
    # l'entrainement (InvalidArgumentError "Unknown image file format"). On teste donc
    # directement avec tf.io.decode_image, le meme decodeur utilise par
    # image_dataset_from_directory en phase 1.2.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            tf.io.decode_image(img_bytes, channels=3)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
IMG_SIZE_TL = (160, 160)
BATCH_SIZE = 32

train_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)
train_ds_tl = train_ds_tl.apply(tf.data.experimental.ignore_errors())

val_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)
val_ds_tl = val_ds_tl.apply(tf.data.experimental.ignore_errors())

# preprocess_input de MobileNetV2 normalise dans [-1, 1] (pas [0, 1]) : pas de
# Rescaling(1./255) ici, preprocess_input le remplace.
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds_tl = train_ds_tl.map(lambda x, y: (preprocess_input(x), y))
val_ds_tl = val_ds_tl.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds_tl = train_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_tl = val_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(160, 160, 3))
x = base_model(inputs, training=False)  # training=False : BatchNorm reste en mode inference
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_tl = tf.keras.Model(inputs, outputs)

In [ ]:
import matplotlib.pyplot as plt


def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history["loss"], label="Train loss")
    ax1.plot(history.history["val_loss"], label="Val loss")
    ax1.set_title(f"{title} - Loss")
    ax1.legend()

    ax2.plot(history.history["accuracy"], label="Train accuracy")
    ax2.plot(history.history["val_accuracy"], label="Val accuracy")
    ax2.set_title(f"{title} - Accuracy")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f"curves_{title.lower().replace(' ', '_').replace('+', '')}.png", dpi=100)
    plt.show()

In [ ]:
# Reprise phase 3.2 : entrainement de la tete (base gelee)
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history_tl_head = model_tl.fit(
    train_ds_tl,
    epochs=10,
    validation_data=val_ds_tl,
)
print(f"val_accuracy finale (tete seule) : {max(history_tl_head.history['val_accuracy']):.3f}")

## Phase 3.3 : Fine-tuning des dernières couches

**Objectif** : dégeler les dernières couches de MobileNetV2, réentraîner avec un learning rate réduit, et gagner 2 à 5 points de `val_accuracy` supplémentaires.

Les premières couches (bords, textures basiques) sont universelles : pas besoin de les retoucher. Les dernières couches (formes complexes, concepts) sont adaptées au domaine cible.

In [ ]:
base_model.trainable = True

print(f"Nombre de couches dans la base : {len(base_model.layers)}")

# Heuristique : geler les 80% premieres couches, degeler les 20% dernieres.
fine_tune_at = int(len(base_model.layers) * 0.8)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Couches degelees : {len(base_model.layers) - fine_tune_at}")
trainable_count = sum(tf.size(w).numpy() for w in model_tl.trainable_weights)
print(f"Parametres entrainables : {trainable_count:,}")

In [ ]:
import datetime, time

# lr x10 plus petit que pour la tete seule : les poids de la base encodent des
# representations valides d'ImageNet. Un grand lr les ecraserait (catastrophic forgetting).
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

log_dir_tl_ft = "logs/transfer/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "-finetune"
tensorboard_callback_tl_ft = tf.keras.callbacks.TensorBoard(log_dir=log_dir_tl_ft, histogram_freq=1)

early_stopping_tl = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

start = time.time()

history_tl_finetune = model_tl.fit(
    train_ds_tl,
    epochs=15,
    validation_data=val_ds_tl,
    callbacks=[tensorboard_callback_tl_ft, early_stopping_tl],
)

training_time_finetune = time.time() - start
print(f"Temps d'entrainement (fine-tuning) : {training_time_finetune:.0f}s")
print(f"val_accuracy finale : {max(history_tl_finetune.history['val_accuracy']):.3f}")

In [ ]:
plot_history(history_tl_finetune, "Transfer Learning - fine-tuning")

In [ ]:
import json as _json

with open("history_tl_finetune.json", "w") as f:
    _json.dump(history_tl_finetune.history, f)
model_tl.save("model_tl.keras")
print("history_tl_finetune.json et model_tl.keras sauvegardes.")

### Vérifications avant la phase 3.4

- **Happy path** : `val_accuracy` gagne 2-5 points par rapport à la phase 3.2, pas de divergence, courbes plus stables que TP1.
- **Edge case** : dégeler 100% des couches avec `lr=1e-5` reste théoriquement valide mais risque réel de détruire les premières couches — à comparer avec le dégelage partiel (80% gelées) si tu veux creuser.
- **Adversarial** : dégeler avec `learning_rate=1e-3` (pas réduit, le même que pour la tête) casserait le pretrained en 2 epochs, la `val_acc` chuterait sous la baseline phase 3.1. C'est exactement pourquoi on baisse le lr pour le fine-tuning.

**Prochaine étape** : Phase 3.4 — tableau de comparaison cross-modèles (scratch / augmenté / transfer learning).